<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 06 · 先保存经验，再审核一次改进

金额转换可能把 1.999 元静默截断。我们先运行检查并整理结论，直接保存一条 Experience。接下来提出一个有证据的改进，通过 Candidate 审核更新同一份经验，观察批准前后哪个版本可以被使用。

**完成后你能做到：** 直接创建和读取 Experience，给现有制品提交 replacement Candidate，批准精确版本，并保留旧版与拒绝不支持的推断。

预计 20 分钟。先按 [README](README.md) 安装环境；本篇可以独立运行，不依赖其他 Notebook 的变量或数据。本篇无需模型和 API Key。

按顺序读说明、运行代码，再对照结果。练习可以改输入；完整重跑使用 **Restart Kernel & Run All**。

## 准备本篇实验

这格启动一个回环地址的真实 Server，并创建独立 Scope，默认把数据保存在本篇自己的 SQLite 文件中，也可按 [README](README.md#使用-oceanbase-运行) 显式选择专用 OceanBase 测试库。
`_tutorial.py` 只管理环境和显示结果；下面的业务调用都是可在应用中复用的公开 API。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))


if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("06", features=())
client = lab.client
assert client is not None

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 06",
        summary="第 06 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-06",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 从真实问题和检查开始

先执行旧写法，观察三位小数被截断。随后使用小型解析函数检查六个输入。
这组用例是教学检查，不能据此宣称覆盖所有货币格式或规模。

In [ ]:
from decimal import Decimal, InvalidOperation

old_value = int(Decimal("1.999") * 100)
print("旧写法把 1.999 转换成了", old_value, "分。")


def parse_amount(text):
    try:
        value = Decimal(text)
    except InvalidOperation as error:
        raise ValueError("amount 必须是数字") from error
    if not value.is_finite() or value < 0 or value * 100 != (value * 100).to_integral_value():
        raise ValueError("amount 必须是非负且最多两位小数的有限金额")
    return int(value * 100)


cases = [
    ("12.34", 1234),
    ("0.01", 1),
    ("1.999", "rejected"),
    ("-1", "rejected"),
    ("NaN", "rejected"),
    ("oops", "rejected"),
]
checked = []
for text, expected in cases:
    try:
        actual = parse_amount(text)
    except ValueError:
        actual = "rejected"
    checked.append({"输入": text, "预期": expected, "实际": actual, "通过": actual == expected})
table(checked)
assert all(row["通过"] for row in checked)

## 2. 把已经确定的经验直接保存

我们已经读过实际检查结果，可以由应用直接提交 Experience。`create_artifact` 接收 family 与内容，
原子创建第一版和本次直接写入的系统来源。`get_artifact` 随后返回完整内容。

这里没有 Candidate，也没有批准步骤。直接写入的来源证明提交了什么内容，不等于系统自动验证了测试结果。

In [ ]:
from powercontext.http import ArtifactReference, CreateArtifactRequest, PrepareContextRequest

initial_content = {
    "situation": "amount: Decimal 金额直接转整数分时，三位小数可能被静默截断。",
    "action": "先检查金额精度，再转成整数分，并用正常和超精度输入验证。",
    "outcome": "本篇的正常输入按预期转换，1.999 被明确拒绝。",
    "lesson": "amount: 先校验金额精度，再转换为整数分。",
}
created = await client.create_artifact(
    scope_id,
    CreateArtifactRequest.model_validate({
        "family": "experience",
        "content": initial_content,
    }),
)
experience_ref = ArtifactReference(family="experience", artifact_id=created.artifact_id, revision=created.revision)
initial = await client.get_artifact(scope_id, "experience", created.artifact_id)
assert initial is not None and initial.content == initial_content
ready = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=5000))
assert ready.content is not None and created.artifact_id in ready.content
show({"直接创建的版本": initial.revision, "经验": initial.content["lesson"], "已进入本次上下文": True})

## 3. 改进结论时，把检查证据一起交给审核者

六个用例还覆盖了负数、NaN 和非法文本。我们希望把经验补充成“校验精度、有限值和非负范围”。
先用统一 Source API 保存实际检查表，再用 `propose_experience` 为已有制品提交改进候选。

`target` 指向刚才的精确 Experience，`source_refs` 指向检查记录。候选的来源关系因此可以供审核者核查。

In [ ]:
from powercontext.http import CreateSourceRequest, ExperienceProposal, ProposeExperienceRequest, SourceReference

evidence = await client.create_source(scope_id, CreateSourceRequest(content={"executed_cases": checked}))
evidence_ref = SourceReference(name="content", source_id=evidence.source_id)
proposal = ExperienceProposal(**{
    **initial_content,
    "action": "转换前检查有限值、非负值和最多两位小数，再运行六个输入用例。",
    "outcome": "六个示例检查通过；超精度、负数、NaN 和非法文本被明确拒绝。",
    "lesson": "amount: 先校验精度、有限值和非负范围，再转整数分；用边界用例验证拒绝行为。",
})
candidate = await client.propose_experience(
    ProposeExperienceRequest(
        scope_id=scope_id,
        proposal=proposal,
        target=experience_ref,
        source_refs=[evidence_ref],
        artifact_refs=[experience_ref],
        reason="根据实际用例补充经验的适用范围",
    )
)
assert candidate.status == "pending" and candidate.result_artifact is None
show(candidate.proposal)

## 4. 有 pending 改进，不等于当前内容已经变化

在审核前重新读取当前制品。它仍应为第一版，内容不变。
`prepare_context` 仍可召回已经提交的旧版；pending 候选本身不会替换它。

In [ ]:
before = await client.get_artifact(scope_id, "experience", created.artifact_id)
assert before is not None and before.revision == 1 and before.content == initial_content
before_context = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=5000))
assert before_context.content is not None
assert initial_content["lesson"] in before_context.content
assert proposal.lesson not in before_context.content
show({"候选状态": candidate.status, "当前制品版本": before.revision, "仍使用的经验": before.content["lesson"]})

## 5. 检查内容后，批准这个精确候选版本

读完上面的改进和实际用例，再执行这格。教程只批准我们自己展示的合成材料；真实应用应由有权限的审核者做决定。
`expected_version` 确保决定针对刚刚检查的候选，批准后才会提交同一制品的后续版本。

## 补充实验：审核期间，候选也会变化

审核者认为内容可以更明确，先修订同一 Candidate。Candidate version 与 Artifact revision 是两套版本：候选修改后，当前制品仍未变化；对旧候选版本的批准必须冲突，避免批准自己没有看过的内容。

In [ ]:
from powercontext.client import ServerResponseError
from powercontext.http import (
    ApproveArtifactCandidateRequest,
    ListArtifactCandidatesRequest,
    ReviseArtifactCandidateRequest,
)

stale_version = candidate.version
proposal = proposal.model_copy(update={"lesson": proposal.lesson + " 本次证据范围限定为上述六个输入。"})
candidate = await client.revise_artifact_candidate(
    ReviseArtifactCandidateRequest(
        scope_id=scope_id,
        candidate_id=candidate.candidate_id,
        expected_version=stale_version,
        proposal=proposal,
        source_refs=[evidence_ref],
        artifact_refs=[experience_ref],
        target=experience_ref,
        reason="审核时明确验证范围",
    )
)
assert candidate.version > stale_version and candidate.status == "pending"
inbox = await client.list_artifact_candidates(ListArtifactCandidatesRequest(scope_id=scope_id))
assert any(
    item.candidate_id == candidate.candidate_id and item.version == candidate.version for item in inbox.candidates
)
try:
    await client.approve_artifact_candidate(
        ApproveArtifactCandidateRequest(
            scope_id=scope_id, candidate_id=candidate.candidate_id, expected_version=stale_version
        )
    )
except ServerResponseError as error:
    assert error.status_code == 409
    print("旧候选版本不能批准；请检查下面的当前提案。")
else:
    raise AssertionError("候选变化后不应允许旧版本审批")
assert (await client.get_artifact(scope_id, "experience", created.artifact_id)).revision == experience_ref.revision
show(candidate.proposal)

In [ ]:
from powercontext.http import ApproveArtifactCandidateRequest

approved = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id,
        candidate_id=candidate.candidate_id,
        expected_version=candidate.version,
    )
)
assert approved.result_artifact is not None and approved.result_artifact.artifact_id == created.artifact_id
current = await client.get_artifact(scope_id, "experience", created.artifact_id)
historical = await client.get_artifact_revision(scope_id, "experience", created.artifact_id, 1)
assert current is not None and current.revision > historical.revision
assert historical.content == initial_content and current.content["lesson"] == proposal.lesson
after_context = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=5000))
assert after_context.content is not None and proposal.lesson in after_context.content
table([
    {"读取方式": "精确历史版本", "Revision": historical.revision, "经验": historical.content["lesson"]},
    {"读取方式": "当前版本", "Revision": current.revision, "经验": current.content["lesson"]},
])

## 练习：拒绝超出证据的结论

六个用例能否支持“覆盖所有货币格式和输入规模”？读下面的候选，再给出拒绝理由。
拒绝后，当前 Experience 的版本与内容应该保持不变。

In [ ]:
from powercontext.http import RejectArtifactCandidateRequest

exaggerated = await client.propose_experience(
    ProposeExperienceRequest(
        scope_id=scope_id,
        proposal=proposal.model_copy(update={"lesson": "此实现已经覆盖所有货币格式和所有输入规模。"}),
        source_refs=[evidence_ref],
        artifact_refs=[],
        reason="练习：检查超出证据范围的推断",
    )
)
show(exaggerated.proposal)
rejected = await client.reject_artifact_candidate(
    RejectArtifactCandidateRequest(
        scope_id=scope_id,
        candidate_id=exaggerated.candidate_id,
        expected_version=exaggerated.version,
        reason="六个用例不能支持所有货币格式和输入规模；请缩小结论或补充验证。",
    )
)
assert rejected.status == "rejected" and rejected.result_artifact is None

after_rejection = await client.get_artifact(scope_id, "experience", created.artifact_id)
assert after_rejection is not None and after_rejection.revision == current.revision

## 保存收获，关闭连接

直接创建让已确定的经验马上可用；Candidate 让待判断的改进保持 pending。两条写入路径最终都形成可按同一套 API 读取的制品版本。

下面关闭本篇 Client 和 Server，保留实验文件供检查。中途停止时也可运行这一格；清理方式见 [README](README.md#清理实验数据)。

下一篇：[07](07_managed_skill.ipynb)。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")